# Inference Test — All Providers

In [1]:
from pathlib import Path
from unified_local_llm_server import LLMProviderPool
from unified_local_llm_server.provider_registry import ProviderRegistry

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
registry = ProviderRegistry.load(ROOT / "providers.example.yaml")
server = LLMProviderPool(provider_registry=registry)

## Provider Status

In [2]:
statuses = {}
for name in server.get_providers():
    statuses[name] = await server.check_provider(name)
    ok = statuses[name]["ok"]
    url = statuses[name]["server_url"]
    print(f"  {'✓' if ok else '✗'} {name:12} {url}")

  ✓ llama_cpp    http://127.0.0.1:9090
  ✓ lm_studio    http://127.0.0.1:1234
  ✓ ollama       http://127.0.0.1:11434
  ✓ unsloth      http://127.0.0.1:8899


## Model Selection

Edit preferred models here. `None` = auto-resolve from `/v1/models`.

In [3]:
PREFERRED_MODELS = {
    "ollama":    "gpt-oss:20b",
    "lm_studio": "openai/gpt-oss-20b",
    "unsloth":   "unsloth/gpt-oss-20b-GGUF@UD-Q4_K_XL",
    "llama_cpp": "gpt-oss-20b-MXFP4",
}

PROMPT = "Return one short sentence about why provider abstraction is useful for local LLMs."
OPTIONS = {"num_predict": 128}

## Run Inference — All Providers

Unloads each provider before switching to the next.

In [4]:
results = {}
previous_provider = None
import time

server.unload_all_models(None)

for provider in server.get_providers():
    if not statuses[provider]["ok"]:
        print(f"[{provider}] SKIP — not reachable")
        continue

    # Unload previous before switching



    # Resolve model
    provider_server = server.get_provider(provider)
    try:
        model = PREFERRED_MODELS.get(provider) or await provider_server.resolve_call_model(None)
    except Exception as exc:
        print(f"[{provider}] SKIP — model resolve failed: {exc}")
        continue

    print(f"\n{'='*60}")
    print(f"  {provider} / {model}")
    print(f"{'='*60}")

    llm = server.load_model(provider, model,context_length=11100)
    try:
        response = await llm.call(
            think=True,
            messages=[{"role": "user", "content": PROMPT}],
            options=OPTIONS,
        )
        print("###################################################")
        time.sleep(3)
        results[provider] = {"model": model, "response": response, "ok": True}
        print(f"Response: {response}")
        previous_provider = provider
    except Exception as exc:
        results[provider] = {"model": model, "error": str(exc), "ok": False}
        print(f"ERROR: {exc}")

    # Cleanup
    if previous_provider:
        try:
            unloaded = server.unload_all_models(previous_provider)
            print(f"\n[{previous_provider}] unloaded: {unloaded}")
        except Exception as exc:
            print(f"[{previous_provider}] unload error: {exc}")


  llama_cpp / gpt-oss-20b-MXFP4
###################################################
Response: <think>We need to return a short sentence about why provider abstraction is useful for local LLMs. Provide a concise sentence. Probably: "Provider abstraction lets local LLMs switch between APIs or local backends seamlessly, simplifying deployment and testing." Let's produce that.</think>

Provider abstraction lets local LLMs switch between APIs or local backends seamlessly, simplifying deployment and testing.

[llama_cpp] unloaded: {'unloaded': ['gpt-oss-20b-MXFP4.gguf']}

  lm_studio / openai/gpt-oss-20b
###################################################
Response: <think>Need a short sentence.</think>

Provider abstraction lets developers swap between local and cloud‑based LLM backends without changing application code, speeding up experimentation and deployment.

[lm_studio] unloaded: {'unloaded': ['openai/gpt-oss-20b']}

  ollama / gpt-oss:20b
############################################

## Summary

In [5]:
print(f"{'Provider':<12} {'Model':<45} {'Status'}")
print("-" * 70)
for provider, r in results.items():
    status = "OK" if r["ok"] else f"FAIL: {r.get('error', '')[:30]}"
    print(f"{provider:<12} {r['model']:<45} {status}")

Provider     Model                                         Status
----------------------------------------------------------------------
llama_cpp    gpt-oss-20b-MXFP4                             OK
lm_studio    openai/gpt-oss-20b                            OK
ollama       gpt-oss:20b                                   OK
unsloth      unsloth/gpt-oss-20b-GGUF@UD-Q4_K_XL           OK
